In [ ]:
from sklearn.svm import LinearSVR
from sklearn.metrics import RocCurveDisplay
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_predict
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score,  mean_absolute_error
import pickle
from scipy.stats import zscore

# Get the cleaned dataset and separate the year and month from the date.
cleaned_data = pd.read_csv("cleaned_dataset.csv")
cleaned_data['Date'] = pd.to_datetime(cleaned_data['Date'], format='ISO8601')
cleaned_data['year'] = cleaned_data['Date'].dt.year
cleaned_data['month'] = cleaned_data['Date'].dt.month
cleaned_data = cleaned_data.drop(columns=['Date'])

# Drop rows with nan values and set up the target.
cleaned_data.dropna(subset=['Retard moyen de tous les trains au départ',"Retard moyen de tous les trains à l'arrivée"], inplace=True)
cleaned_data.dropna(inplace=True)

# Using the Z-score, outliers are removed when they are apart of the rest.
cleaned_data['ZScore'] = zscore(cleaned_data["Retard moyen de tous les trains au départ"])
cleaned_data = cleaned_data[cleaned_data["ZScore"].abs() <= 3]
cleaned_data['ZScoreA'] = zscore(cleaned_data["Retard moyen de tous les trains à l'arrivée"])
cleaned_data = cleaned_data[cleaned_data["ZScoreA"].abs() <= 3]


# Use a mean of delays' means.
cleaned_delay = ((cleaned_data['Retard moyen de tous les trains au départ'] + cleaned_data["Retard moyen de tous les trains à l'arrivée"])/2)


# Convert objects to categories.
for col in cleaned_data.select_dtypes(include='object').columns:
    cleaned_data[col] = cleaned_data[col].astype('category').cat.codes

# Initialize X and y.
X = cleaned_data.drop(["Retard moyen de tous les trains au départ","Retard moyen de tous les trains à l'arrivée",], axis=1).values
y = cleaned_delay

    #cv_linear = KFold(n_splits=i, shuffle=True, random_state=42)
    #linear_model = make_pipeline(StandardScaler(),LinearSVR(max_iter=500, random_state=42))
    #linear_predictions = cross_val_predict(linear_model,X, y, cv=cv_linear)
    #print(mean_absolute_error(y,linear_predictions),r2_score(y,linear_predictions),mean_squared_error(y,linear_predictions))

#Model setup.
i = 2
cv_tree = KFold(n_splits=5, shuffle=True, random_state=42)
tree_model = RandomForestRegressor(n_estimators=500, random_state=42)
# By cross-validating, we ensure the model is trained correctly.
tree_predictions = cross_val_predict(tree_model,X, y, cv=cv_tree)
# Training the model.
tree_model.fit(X, y)
print(mean_absolute_error(y,tree_predictions),r2_score(y,tree_predictions),mean_squared_error(y,tree_predictions))

#Create model file.
with open('model.pkl','wb') as output:
    pickle.dump(tree_model,output,pickle.HIGHEST_PROTOCOL)

/tmp/ipykernel_26423/1538309278.py:38: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in cleaned_data.select_dtypes(include='object').columns:


In [ ]:

import matplotlib.pyplot as plt
pred = tree_model.predict(X)
plt.scatter(y, pred, alpha=0.5)
plt.xlabel("True delay")
plt.ylabel("Predicted delay")
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')
plt.show()
print(pred)